In [2]:
!pip install pydub

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import plotly.express as px
from scipy.signal import find_peaks
import os
from pydub import AudioSegment


# custom functions to perfomr Leq and db sum
def sum_levels(levels):
    l = np.array(levels)
    return 10*np.log10(np.sum(np.power(10,l/10)))

def leq(levels):
    l = np.array(levels)
    return 10*np.log10(np.mean(np.power(10,l/10)))

c:\Users\scjaa\anaconda3\envs\tensorflow_gpu_11_2\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [23]:
# opne csv file
csv_file = r"\\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT\20231211_SANTUR\5-Resultados\P1_CONTENEDORES\SPL\leq_levels_P1_CONTENEDORES_v1_1_test.csv"
df = pd.read_csv(csv_file)
title = csv_file.split("\\")[-3]

# get the audiomoth folder
audiomoth_folder = csv_file.replace("5-Resultados","3-Medidas").replace("SPL","AUDIOMOTH")
# remove last item from audio moth folder
audiomoth_folder = "\\".join(audiomoth_folder.split("\\")[:-1])

output_folder = csv_file.split("\\")[:-1]
output_folder = os.path.join(output_folder)
output_folder = os.path.join(*output_folder, "peak_clips")

os.makedirs(output_folder, exist_ok=True)

# # chagen the filename to the full path
df['filename'] = df['filename'].apply(lambda x: os.path.join(audiomoth_folder, x))
df

TypeError: join() argument must be str, bytes, or os.PathLike object, not 'list'

In [5]:
# convert date column in datetime
df['date'] = pd.to_datetime(df['date'])

duration = df['date'].iloc[-1] - df['date'].iloc[0]
duration = duration.total_seconds()
print(f"Duration: {duration} seconds, {duration/60} minutes, {duration/3600} hours, {duration/3600/24} days")

la = df['LA'].values
print('\nMax:', np.max(la).round(2))
print('Min:', np.min(la).round(2))

print('\nMediana:', np.median(la).round(2))
print('Promedio:', leq(la).round(2))
print('Standard deviation:', np.std(la).round(2))

print('\nPercentil 98:', np.quantile(la, 0.99).round(2)) # percentil 98 means that 98% of the data is below this value

Duration: 749819.0 seconds, 12496.983333333334 minutes, 208.28305555555556 hours, 8.678460648148148 days

Max: 96.9
Min: 55.77

Mediana: 67.47
Promedio: 72.69
Standard deviation: 5.5

Percentil 98: 82.8


In [16]:
def save_clips_from_peaks(df, output_folder, sampling_rate=44100):
    # if output folder does not exist, create it
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    for index, row in df.iterrows():
        audio_file = row['filename']
        start_sample = row['start_time']
        end_sample = row['end_time']
        
        # milliseconds
        start_time_ms = int((start_sample / sampling_rate) * 1000)
        end_time_ms = int((end_sample / sampling_rate) * 1000)
        
        # Load the audio file
        audio = AudioSegment.from_file(audio_file)
        clip = audio[start_time_ms : end_time_ms]
        
        # convert output folder to a path
        output_folder = os.path.join('\\\\', output_folder)
        # get the last item of the filename
        filename = audio_file.split("\\")[-1]
        clip_path = os.path.join(output_folder, f"peak_{filename}_{index}.wav")
        print(f"Saving clip {clip_path}")
        clip.export(clip_path, format='wav')
        

window_size = 30  # seconds
prominence = 1
width = 1

df['LA_median'] = df['LA'].rolling(window=window_size, min_periods=1).quantile(0.5) + 10

# go through each day
start_date = df['date'].dt.normalize().iloc[0]
end_date = df['date'].dt.normalize().iloc[-1]
current_date = start_date

peaks_num = []
while current_date <= end_date:
    daily_data = df[(df['date'] >= current_date) & (df['date'] < current_date + pd.Timedelta(days=1))]
    
    #dynamic threshold by filtering data points
    above_threshold = daily_data[daily_data['LA'] > daily_data['LA_median']]
    
    #find peaks in the filtered data
    if not above_threshold.empty:
        peaks, properties = find_peaks(above_threshold['LA'], prominence=prominence, width=width)
        peak_filenames = above_threshold.iloc[peaks]['filename'].values
        start_times = properties['left_ips']
        end_times = properties['right_ips']
        
        # Creating DataFrame for each peak
        peaks_df = pd.DataFrame({
            'filename': peak_filenames,
            'start_time': start_times,
            'end_time': end_times
        })
            
        print(peaks_df)

        save_clips_from_peaks(peaks_df, output_folder)        

                                             filename  start_time    end_time
0   \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...    3.951515    6.495726
1   \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...    8.656051   10.528846
2   \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...   12.795872   15.984375
3   \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...   19.738095   22.548583
4   \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...   26.588529   29.898785
..                                                ...         ...         ...
70  \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...  387.500000  388.890625
71  \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...  391.335586  393.889488
72  \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...  397.197561  403.407671
73  \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...  406.052381  409.713018
74  \\192.168.205.117\AAC_Server\PUERTOS\NOISEPORT...  414.785185  416.236842

[75 rows x 3 columns]
Saving clip \\192.168.205.117\AAC_Server\

FileNotFoundError: [Errno 2] No such file or directory: '\\\\192.168.205.117\\AAC_Server\\PUERTOS\\NOISEPORT\\20231211_SANTUR\\5-Resultados\\P1_CONTENEDORES\\SPL\\peak_clips\\peak_20231211_140420.WAV_0.wav'